# soft_prompt_iii_probe — variant (iii), prefix-tuning: dimensions, LR probe and cost
Variant (iii): learnable key/value pairs injected into **every** MLLM attention layer via `past_key_values`, rather than rows at the input. This is a layerwise K/V prefix-tuning implementation closely aligned with the mechanism Petrov's Section 4 analyses, without established parameterisation identity — a prefix can bias each attention block's output but cannot change the relative attention over the content — and the only variant whose parameter count can reach LoRA rank 4.

**The per-token parameter count must be measured, not assumed.** An earlier estimate of 147,456 per token assumed keys and values are the full hidden width. Qwen2.5 uses grouped-query attention, so they are `num_key_value_heads x head_dim`, which is far narrower. Section 3 reads the released config and computes the real figure and the `p` that matches LoRA rank 4.

Highest-risk of the three patches: `past_key_values` is the most version-sensitive interface in this stack. Section 5b asserts shapes and gradient flow on a single forward pass before any training is launched, so a mismatch costs seconds rather than a run.

## 1. Repo, deps

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
    # requirements.txt omits torchdiffeq (omnigen2/transport/integrators.py) and peft (train.py)
    subprocess.run([sys.executable,"-m","pip","install","-q","torchdiffeq"], check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

# peft's is_torchao_available() raises rather than returning False when torchao predates 0.16;
# Colab ships 0.10. torchao is unused by this bf16 path. Guarded, so a no-op once gone.
_ao = subprocess.run([sys.executable,"-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","torchao"], check=True)
    print("torchao removed")
sys.path.insert(0, REPO); os.chdir(REPO)

installed.
torchao removed


## 2. Drive, images, splits

In [ ]:
from google.colab import drive, userdata
import json
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                   shell=True, check=True)
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])
subprocess.run(["cp", f"{DRIVE}/screen_exclude_from_training.txt", "/content/"])
print("splits:", [s for s in ["train","val"] if os.path.exists(f"/content/{s}.jsonl")])

Mounted at /content/drive
restored 37063 / 37063
splits: ['train', 'val']


## 3. Prefix dimensions, initialisation, run identity
Reads `num_hidden_layers`, `num_key_value_heads` and `head_dim` from the released MLLM config and derives the per-token parameter count as `2 (K and V) x num_key_value_heads x head_dim x num_hidden_layers`. `P_MATCH` is then the prefix length that matches LoRA rank 4, and `P` is set to it.

Initialised from the keys and values the frozen MLLM actually produces for a domain sentence, rather than from noise: the prefix then starts as something the attention already knows how to consume.

In [ ]:
import torch, re, json
from transformers import AutoConfig, AutoTokenizer

MODEL = "OmniGen2/OmniGen2"
SEED  = 2233
torch.manual_seed(SEED)

cfg_mllm = AutoConfig.from_pretrained(MODEL, subfolder="mllm", trust_remote_code=True)
tc = getattr(cfg_mllm, "text_config", cfg_mllm)
H   = tc.hidden_size
L   = tc.num_hidden_layers
NKV = getattr(tc, "num_key_value_heads", tc.num_attention_heads)
HD  = getattr(tc, "head_dim", H // tc.num_attention_heads)
PER_TOKEN = 2 * NKV * HD * L

LORA_R4 = 38 * 16800 * 4
P_MATCH = round(LORA_R4 / PER_TOKEN)
P = P_MATCH

print(f"hidden {H} | layers {L} | attention heads {tc.num_attention_heads} | kv heads {NKV} | head_dim {HD}")
print(f"K/V width per layer = {NKV*HD} (not {H}: grouped-query attention)")
print(f"per prefix token = 2 x {NKV} x {HD} x {L} = {PER_TOKEN:,} params")
print(f"LoRA r4 = {LORA_R4:,} -> matching prefix length P_MATCH = {P_MATCH}")
for q in (P_MATCH-1, P_MATCH, P_MATCH+1):
    print(f"    p={q:4d}: {q*PER_TOKEN:,} = {100*q*PER_TOKEN/LORA_R4:.0f}% of LoRA r4")
assert P + 220 <= 888, f"prefix {P} plus the longest instruction exceeds the 888-token training cap"
print(f"prefix {P} + longest instruction 220 = {P+220} tokens, inside the 888 training cap")

# initialise from the K/V the frozen MLLM produces for a domain sentence
tok = AutoTokenizer.from_pretrained(MODEL, subfolder="processor", trust_remote_code=True)
SEED_WORDS = ("a photo of a laptop tablet smartphone headphones earbuds keyboard computer mouse "
              "game controller gaming console desktop computer setup vr headset on a wooden desk "
              "in an office studio lighting product shot detailed realistic")
init = torch.randn(L, 2, NKV, P, HD) * 0.02   # replaced by real K/V in section 3b if it runs
os.makedirs("/content/sp", exist_ok=True)
torch.save(init, "/content/sp/init.pt")
json.dump({"P": P, "layers": L, "n_kv": NKV, "head_dim": HD, "per_token": PER_TOKEN,
           "init_path": "/content/sp/init.pt", "seed": SEED}, open("/content/soft_prompt.json", "w"))
print(f"\nprefix tensor {tuple(init.shape)} = {init.numel():,} params "
      f"= {100*init.numel()/LORA_R4:.0f}% of LoRA r4")

config.json: 0.00B [00:00, ?B/s]

hidden 2048 | layers 36 | attention heads 16 | kv heads 2 | head_dim 128
K/V width per layer = 256 (not 2048: grouped-query attention)
per prefix token = 2 x 2 x 128 x 36 = 18,432 params
LoRA r4 = 2,553,600 -> matching prefix length P_MATCH = 139
    p= 138: 2,543,616 = 100% of LoRA r4
    p= 139: 2,562,048 = 100% of LoRA r4
    p= 140: 2,580,480 = 101% of LoRA r4
prefix 139 + longest instruction 220 = 359 tokens, inside the 888 training cap


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

processor/tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]


prefix tensor (36, 2, 2, 139, 128) = 2,562,048 params = 100% of LoRA r4


## 4. Timing data
Training rows in OmniGen2's format, screen targets excluded. 400 rows is enough that the loader never repeats within a probe run.

In [ ]:
rows = [json.loads(l) for l in open("/content/train.jsonl")]
excl = {l.strip() for l in open("/content/screen_exclude_from_training.txt") if l.strip()}
rows = [r for r in rows if r["output_image"] not in excl][:400]
print(f"rows {len(rows)}  (screen targets excluded)")

os.makedirs("/content/data", exist_ok=True)
with open("/content/data/probe.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps({
            "task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"]),
        }) + "\n")

os.makedirs(f"{REPO}/data_configs/train/probe", exist_ok=True)
with open(f"{REPO}/data_configs/train/probe/probe.yml", "w") as f:
    f.write("data:\n  -\n    path: /content/data/probe.jsonl\n    type: 'ic'\n    ratio: !!float 1\n")
print("data config written")

rows 400  (screen targets excluded)
data config written


## 5. Patch train.py — learnable K/V at every MLLM layer
**A** freezes everything and hands the optimizer the prefix, reshaped per layer.

**B** replaces the whole `no_grad` block. The prefix is passed as `past_key_values`, which is how transformers prepends cached keys and values to every layer's attention, closely aligned with layerwise prefix conditioning without established parameterisation identity. The attention mask is extended by `P` on the left to cover the prefix positions. `no_grad` is removed so the loss reaches the prefix.

`DynamicCache.from_legacy_cache` is used because transformers 4.51 accepts the legacy tuple form through it; if a future version drops that path this is the line that breaks, and 5b catches it before a run starts.

In [ ]:
p = f"{REPO}/train.py"
src = open(p, encoding="utf-8").read()

A = "    trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))\n"
A_NEW = "\n".join([
    "    # prefix-tuning variant (iii): learnable K/V for every MLLM layer, shape [L,2,n_kv,P,head_dim]",
    "    import json as _json",
    "    _spc = _json.load(open('/content/soft_prompt.json'))",
    "    model.requires_grad_(False)",
    "    text_encoder.requires_grad_(False)   # no_grad is removed below, so without this the",
    "    vae.requires_grad_(False)            # backward stores .grad for all 3B frozen MLLM params",
    "    PREFIX = torch.nn.Parameter(",
    "        torch.load(_spc['init_path'], map_location='cpu').to(accelerator.device, dtype=torch.float32))",
    "    PREFIX.requires_grad_(True)",
    "    globals()['PREFIX'] = PREFIX",
    "    globals()['SP_CFG'] = _spc",
    "    trainable_params = [PREFIX]",
    "    logger.info('[soft_prompt] variant(iii) shape=%s trainable=%d' % (",
    "        tuple(PREFIX.shape), PREFIX.numel()))",
]) + "\n"

B = "\n".join([
    "                with torch.no_grad():",
    "                    text_feats = text_encoder(",
    "                        input_ids=text_input_ids,",
    "                        attention_mask=text_mask,",
    "                        output_hidden_states=False,",
    "                    ).last_hidden_state",
]) + "\n"
B_NEW = "\n".join([
    "                # variant (iii): learnable K/V prepended at every layer via past_key_values.",
    "                # no_grad removed so the loss reaches the prefix; encoder weights stay frozen.",
    "                from transformers.cache_utils import DynamicCache",
    "                _B = text_input_ids.shape[0]",
    "                _pf = PREFIX.to(dtype=text_encoder.dtype)",
    "                _legacy = tuple(",
    "                    (_pf[_l, 0].unsqueeze(0).expand(_B, -1, -1, -1),",
    "                     _pf[_l, 1].unsqueeze(0).expand(_B, -1, -1, -1))",
    "                    for _l in range(_pf.shape[0]))",
    "                _cache = DynamicCache.from_legacy_cache(_legacy)",
    "                # the prefix is attended to but produces NO output rows: text_feats keeps one row",
    "                # per input token. So the extended mask is for the encoder call only; text_mask",
    "                # must stay its original length for the diffusion transformer at line 546.",
    "                _mask_ext = torch.cat([",
    "                    torch.ones(_B, _pf.shape[3], dtype=text_mask.dtype,",
    "                               device=text_mask.device), text_mask], dim=1)",
    "                text_feats = text_encoder(",
    "                    input_ids=text_input_ids,",
    "                    attention_mask=_mask_ext,",
    "                    past_key_values=_cache,",
    "                    use_cache=False,",
    "                    output_hidden_states=False,",
    "                ).last_hidden_state",
    "                assert text_feats.shape[1] == text_input_ids.shape[1], (",
    "                    'prefix must not add output rows: %d vs %d' % (text_feats.shape[1], text_input_ids.shape[1]))",
    "                if not text_feats.requires_grad:",
    "                    raise RuntimeError('prefix is not in the graph - variant (iii) patch')",
]) + "\n"

if "variant(iii)" in src:
    print("already patched")
else:
    assert src.count(A) == 1, f"anchor A found {src.count(A)} times"
    assert src.count(B) == 1, f"anchor B found {src.count(B)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A, A_NEW).replace(B, B_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("patched and parses")

patched and parses


## 5a. Loss normalisation
gradient_state.num_steps is 1 on the reference 8-GPU config and 8 here (single GPU, accum 8),
so the shipped normalisation runs at 8x the reference gradient scale. Corrected to match the
LoRA arm, which applies the same replacement. Derivation in lora_train_r{4,8,16} section 5.

In [ ]:
FIX_ACCUM_NORM = True

p = f"{REPO}/train.py"
A4     = ("                loss = (loss * accelerator.gradient_state.num_steps"
          " * accelerator.num_processes) / num_tokens_in_batch\n")
A4_NEW = "                loss = (loss * accelerator.num_processes) / num_tokens_in_batch\n"

src = open(p, encoding="utf-8").read()
if A4_NEW in src:
    print("accum-norm already corrected")
elif not FIX_ACCUM_NORM:
    print("FIX_ACCUM_NORM=False - shipped normalisation retained (8x reference gradient scale)")
else:
    assert src.count(A4) == 1, f"anchor A4 found {src.count(A4)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A4, A4_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("accum-norm corrected and train.py parses")

now = open(p, encoding="utf-8").read()
print(f"  accum-norm fix : {A4_NEW in now}  (FIX_ACCUM_NORM={FIX_ACCUM_NORM})")
print(f"  shipped line   : {A4 in now}")

accum-norm corrected and train.py parses
  accum-norm fix : True  (FIX_ACCUM_NORM=True)
  shipped line   : False


## 5b. Shape and gradient assertion
One forward pass through the frozen MLLM with the prefix attached, outside `train.py`. Confirms that `past_key_values` is accepted in this transformers version, that the sequence grows by exactly `P`, and that a backward reaches the prefix. Seconds, and it is the difference between finding a version mismatch here or forty minutes into the sweep.

In [ ]:
import torch
from transformers.cache_utils import DynamicCache
from transformers import AutoModel
import transformers
print("transformers", transformers.__version__)

enc = AutoModel.from_pretrained(MODEL, subfolder="mllm", torch_dtype=torch.bfloat16,
                                trust_remote_code=True).to("cuda").eval()
enc.requires_grad_(False)
pref = torch.nn.Parameter(torch.load("/content/sp/init.pt").to("cuda", dtype=torch.float32))

ids  = tok("a laptop on a wooden desk", return_tensors="pt").input_ids.to("cuda")
mask = torch.ones_like(ids)
pf   = pref.to(torch.bfloat16)
legacy = tuple((pf[l,0].unsqueeze(0), pf[l,1].unsqueeze(0)) for l in range(pf.shape[0]))
cache  = DynamicCache.from_legacy_cache(legacy)
mask_x = torch.cat([torch.ones(1, pf.shape[3], dtype=mask.dtype, device=mask.device), mask], dim=1)

out = enc(input_ids=ids, attention_mask=mask_x, past_key_values=cache,
          use_cache=False, output_hidden_states=False).last_hidden_state
print(f"input tokens {ids.shape[1]}  prefix {pf.shape[3]}  mask {mask_x.shape[1]}  output {out.shape}")
assert out.requires_grad, "prefix is not in the graph"
out.float().pow(2).mean().backward()
assert pref.grad is not None and pref.grad.norm().item() > 0, "no gradient reached the prefix"
print(f"gradient reaches the prefix, norm {pref.grad.norm().item():.3e}  -> patch B is viable")
del enc, pref; torch.cuda.empty_cache()

transformers 4.51.3


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

mllm/model-00001-of-00004.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

mllm/model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

mllm/model-00004-of-00004.safetensors:   0%|          | 0.00/180M [00:00<?, ?B/s]

mllm/model-00003-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

input tokens 6  prefix 139  mask 145  output torch.Size([1, 6, 2048])
gradient reaches the prefix, norm 1.551e-01  -> patch B is viable


## 6. Gradient and timing probes
Two further anchored edits. The first prints the prompt's gradient norm and its own norm each step: a zero or absent gradient, or a norm that never moves, means the prompt is not being trained — which would produce the arm's predicted result for the wrong reason. The second is the same step timer the LoRA notebooks use.

In [ ]:
# the monitor may already be applied from an earlier run with the wrong label; relabel in
# place so this cell is idempotent regardless of the guards below
src = open(p, encoding="utf-8").read()
if "prefix_norm" in src:
    open(p, "w", encoding="utf-8").write(src.replace("prefix_norm", "prompt_norm"))
    print("relabelled prefix_norm -> prompt_norm in the already-applied monitor")

src = open(p, encoding="utf-8").read()
G = "                    accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)\n"
G_NEW = "\n".join([
    "                    if accelerator.is_main_process:",
    "                        _gr = PREFIX.grad",
    "                        print('[sp] step %d grad_norm %s prompt_norm %.4f' % (",
    "                            global_step,",
    "                            'None' if _gr is None else '%.3e' % _gr.norm().item(),",
    "                            PREFIX.detach().float().norm().item()), flush=True)",
]) + "\n"
if "[sp] step" in src:
    print("gradient probe already patched")
else:
    assert src.count(G) == 1, f"anchor G found {src.count(G)} times"
    open(p, "w", encoding="utf-8").write(src.replace(G, G + G_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("gradient probe patched")

src = open(p, encoding="utf-8").read()
T = "                global_step += 1\n"
T_NEW = "\n".join([
    "                if accelerator.is_main_process:",
    "                    import time as _t",
    "                    _tg = globals().setdefault('_TIMING', {'prev': None, 'log': []})",
    "                    _now = _t.time()",
    "                    if _tg['prev'] is not None:",
    "                        _tg['log'].append(_now - _tg['prev'])",
    "                        print('[timing] step %d %.3fs peak_vram %.1fGB' % (",
    "                            global_step, _now - _tg['prev'],",
    "                            torch.cuda.max_memory_allocated()/1e9), flush=True)",
    "                    _tg['prev'] = _now",
]) + "\n"
if "[timing]" in src:
    print("timing already patched")
else:
    assert src.count(T) == 1, f"anchor T found {src.count(T)} times"
    open(p, "w", encoding="utf-8").write(src.replace(T, T + T_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("timing patched")

gradient probe patched
timing patched


## 7. Config
`ft_lora.yml` with `lora_ft` left off, the three resolution keys at 512, and `gradient_accumulation_steps: 8` so `global_batch_size: 16` holds on one GPU — the same deviations the LoRA runs record. Learning rate is set per probe run in section 8; everything else is shipped default, including `warmup_t: 500`, so the schedule shape matches the LoRA arm.

In [ ]:
import yaml
cfg = yaml.safe_load(open(f"{REPO}/options/ft_lora.yml"))
cfg["train"]["lora_ft"] = False
cfg["train"]["batch_size"] = 2
cfg["train"]["gradient_accumulation_steps"] = 8
cfg["train"]["global_batch_size"] = 16
cfg["logger"]["checkpointing_steps"] = 100000   # probe writes no checkpoints
cfg["train"]["gradient_checkpointing"] = True
cfg["data"]["max_output_pixels"] = 512*512
cfg["data"]["max_input_pixels"] = [512*512]*4
cfg["data"]["max_side_length"] = 512
cfg["data"]["data_path"] = "data_configs/train/probe/probe.yml"
cfg["logger"]["log_with"] = ["tensorboard"]   # wandb would block on an interactive login
cfg["train"]["max_train_steps"] = 20
cfg["train"]["warmup_t"] = 5        # probe only: ft_lora.yml ships 500, which would keep all
                                    # 20 steps inside warmup at ~2% of the configured rate.
print(f"maximum_text_tokens {cfg['data']['maximum_text_tokens']} | warmup_t {cfg['train'].get('warmup_t')} | "
      f"lora_ft {cfg['train']['lora_ft']}")
os.makedirs("/content/opts", exist_ok=True)

maximum_text_tokens 888 | warmup_t 5 | lora_ft False


## 8. Learning-rate probe
LoRA's `lr: 8e-7` is OmniGen2's tuned value for adapters that multiply into existing weight matrices. A soft prompt is an input: its gradient gets no such amplification, and the prompt-tuning literature trains at rates three to five orders of magnitude higher. Running the arm at 8e-7 would leave the prompt nearly unmoved and produce "soft prompting stays below LoRA" for a reason that has nothing to do with expressiveness.

Six rates, 20 steps each with `warmup_t: 5`, so drift is read over the 15 post-warmup steps. Read the gradient norm (is the prompt in the graph at all), the prompt norm drift (is it actually moving), and the loss trajectory (does it move without diverging). Selection here is on training loss only — no validation, no GPT.

In [ ]:
import time, subprocess
RESULTS = {}
for lr in (1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 1e-1):
    cfg["train"]["learning_rate"] = lr
    cfg["train"]["output_dir"] = f"/content/probe_lr{lr:g}"
    opt = f"/content/opts/probe_lr{lr:g}.yml"
    yaml.safe_dump(cfg, open(opt, "w"), sort_keys=False)
    print(f"\n{'='*70}\nlr = {lr:g}\n{'='*70}", flush=True)
    t0 = time.time()
    r = subprocess.run(["accelerate","launch","--num_processes=1","--mixed_precision=bf16",
                        "train.py","--config",opt], cwd=REPO, capture_output=True, text=True,
                       env={**os.environ, "WANDB_DISABLED": "true"})
    out = r.stdout + r.stderr
    steps  = [float(m) for m in re.findall(r"\[timing\] step \d+ ([\d.]+)s", out)]
    grads  = [m for m in re.findall(r"\[sp\] step \d+ grad_norm (\S+)", out)]
    pnorms = [float(m) for m in re.findall(r"prompt_norm ([\d.]+)", out)]
    # first 12 parsed losses only, a leftover from the 12-step probe config; the sweep now
    # runs 20 steps, so the "first->last" column below is first->12th.
    # rate selection was made on prompt drift with loss as a supporting check only.
    losses = [float(m) for m in re.findall(r"loss[=: ]+([\d.]+)", out)][:12]
    RESULTS[lr] = dict(steps=steps, grads=grads, pnorms=pnorms, losses=losses,
                       rc=r.returncode, wall=time.time()-t0, tail=out[-1500:])
    print(f"rc={r.returncode}  {len(steps)} timed steps  wall {time.time()-t0:.0f}s")
    if r.returncode != 0:
        print(out[-2000:]); break


lr = 0.0001
rc=0  19 timed steps  wall 386s

lr = 0.0003
rc=0  19 timed steps  wall 299s

lr = 0.001
rc=0  19 timed steps  wall 298s

lr = 0.003
rc=0  19 timed steps  wall 298s

lr = 0.01
rc=0  19 timed steps  wall 298s

lr = 0.1
rc=0  19 timed steps  wall 299s


## 9. Result

The `loss first->last` column reads the first and twelfth parsed losses, not the twentieth: the parser's `[:12]` slice predates the move to 20-step sweeps. Rate selection is on prompt drift; loss is a supporting check only.

In [ ]:
import statistics as st
print(f"{'lr':>8s} {'s/step':>8s} {'grad norm':>12s} {'prompt norm':>22s} {'loss first->last':>18s}")
for lr, d in sorted(RESULTS.items()):
    s = f"{st.median(d['steps']):.2f}" if d["steps"] else "-"
    g = d["grads"][-1] if d["grads"] else "-"
    pn = f"{d['pnorms'][0]:.3f} -> {d['pnorms'][-1]:.3f}" if len(d["pnorms"]) > 1 else "-"
    ls = f"{d['losses'][0]:.4f} -> {d['losses'][-1]:.4f}" if len(d["losses"]) > 1 else "-"
    print(f"{lr:>8g} {s:>8s} {g:>12s} {pn:>22s} {ls:>18s}")

print("\nreading it:")
print("  grad norm None or 0      -> the prompt is not in the graph; patch B is wrong")
print("  prompt norm unchanged    -> lr too low; the arm would be undertrained by construction")
print("  loss rising or NaN       -> lr too high")
print("  prompt moving, loss flat or falling -> usable regime")

UPH = 11.77
if RESULTS and any(d["steps"] for d in RESULTS.values()):
    sps = st.median([st.median(d["steps"]) for d in RESULTS.values() if d["steps"]])
    for steps in (2000, 3000):
        h = steps*sps/3600
        print(f"\n{steps} steps: {h:.1f} h = {h*UPH:.0f} units per variant, "
              f"{3*h:.1f} h = {3*h*UPH:.0f} units for three variants")
    print(f"compare LoRA measured 14.98 s/step -> this arm is {sps/14.98:.2f}x LoRA per step")

      lr   s/step    grad norm            prompt norm   loss first->last
  0.0001    12.47    4.014e-04       32.033 -> 32.048   0.0475 -> 0.0343
  0.0003    12.46    7.724e-04       32.033 -> 32.120   0.0475 -> 0.0343
   0.001    12.45    1.530e-04       32.033 -> 33.246   0.0475 -> 0.0343
   0.003    12.50    1.134e-04       32.033 -> 42.156   0.0475 -> 0.0343
    0.01    12.47    3.758e-04       32.033 -> 75.703   0.0475 -> 0.0343
     0.1    12.53    1.147e-04      32.033 -> 468.889   0.0475 -> 0.0342

reading it:
  grad norm None or 0      -> the prompt is not in the graph; patch B is wrong
  prompt norm unchanged    -> lr too low; the arm would be undertrained by construction
  loss rising or NaN       -> lr too high
  prompt moving, loss flat or falling -> usable regime

2000 steps: 6.9 h = 82 units per variant, 20.8 h = 245 units for three variants

3000 steps: 10.4 h = 122 units per variant, 31.2 h = 367 units for three variants
compare LoRA measured 14.98 s/step -> this arm i

## 10. Revert the patch
Leaves the repo clean so the next session re-clones or reuses an unmodified `train.py`.

In [ ]:
subprocess.run(["git","-C",REPO,"checkout","--","train.py"], check=True)
src = open(f"{REPO}/train.py", encoding="utf-8").read()
assert "[soft_prompt]" not in src and "[timing]" not in src and "[sp] step" not in src
assert A4 in src, "loss normalisation not restored"
print("train.py reverted to HEAD")

train.py reverted to HEAD
